In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os, time, json
from tqdm import tqdm

# --- 1. Configuration & Stats ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RAW = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw'
TEST = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in'
MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
FEATS = ['cpm25', 'u10', 'v10', 'pblh', 't2'] # Added Temp for better chemistry logic

print("Computing Z-Score Stats...")
stats_mean, stats_std = [], []
for f in FEATS:
    all_vals = np.concatenate([np.load(f'{RAW}/{m}/{f}.npy').flatten() for m in MONTHS])
    stats_mean.append(all_vals.mean())
    stats_std.append(all_vals.std() + 1e-7)
stats = [np.array(stats_mean), np.array(stats_std)]

# --- 2. Advanced Dataset ---
class PollutionDeltaDataset(Dataset):
    def __init__(self, split='train'):
        self.data = {f: np.concatenate([np.load(f'{RAW}/{m}/{f}.npy') for m in MONTHS], axis=0) for f in FEATS}
        T_total = self.data['cpm25'].shape[0]
        indices = list(range(T_total - 26))
        np.random.seed(42)
        np.random.shuffle(indices)
        split_idx = int(len(indices) * 0.85)
        self.idx_list = indices[:split_idx] if split == 'train' else indices[split_idx:]

    def __len__(self): return len(self.idx_list)

    def __getitem__(self, i):
        start = self.idx_list[i]
        # Normalize all features
        X = np.stack([(self.data[f][start:start+10] - stats[0][j])/stats[1][j] for j, f in enumerate(FEATS)], axis=-1)
        # Target: Change in normalized PM2.5 from the last input hour
        Y_target = (self.data['cpm25'][start+10:start+26] - stats[0][0])/stats[1][0]
        last_pm = X[-1, :, :, 0] 
        Y_delta = Y_target - last_pm 
        
        # Reshape to (Channels*Time, H, W) -> (50, 140, 124)
        X = X.transpose(3, 0, 1, 2).reshape(-1, 140, 124)
        return torch.from_numpy(X).float(), torch.from_numpy(Y_delta).float()

# --- 3. The "Better" Model: Multi-Scale Residual UNet ---
class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv3x3 = nn.Conv2d(in_ch, out_ch//2, 3, padding=1)
        self.conv5x5 = nn.Conv2d(in_ch, out_ch//2, 5, padding=2)
        self.norm = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return F.gelu(self.norm(torch.cat([self.conv3x3(x), self.conv5x5(x)], dim=1)))

class PollutionUNet(nn.Module):
    def __init__(self, in_channels=50, out_channels=16):
        super().__init__()
        # Encoder
        self.enc1 = MultiScaleBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = MultiScaleBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        # Bridge
        self.bridge = MultiScaleBlock(128, 256)
        
        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = MultiScaleBlock(256, 128) # Concatenation makes it 256
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = MultiScaleBlock(128, 64)
        
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        # Handle odd dimensions (140x124) by padding to 144x128 for symmetric UNet
        x = F.pad(x, (2, 2, 2, 2)) 
        
        h1 = self.enc1(x)
        h2 = self.enc2(self.pool1(h1))
        b  = self.bridge(self.pool2(h2))
        
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, h2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, h1], dim=1))
        
        out = self.final(d1)
        return out[:, :, 2:-2, 2:-2] # Crop back to 140x124

model = PollutionUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=200, epochs=20)

# --- 4. Training ---
train_loader = DataLoader(PollutionDeltaDataset('train'), batch_size=8, shuffle=True, num_workers=4)
scaler = torch.amp.GradScaler('cuda')

print("Starting Fast Training...")
for epoch in range(20):
    model.train()
    t_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x)
            loss = F.huber_loss(pred, y) # Huber is more robust to pollution spikes than MSE
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {t_loss/len(train_loader):.5f}")

# --- 5. Final Inference ---
print("Running Inference...")
model.eval()

t_c = np.load(os.path.join(TEST, "cpm25.npy")).astype(np.float32)
t_u = np.load(os.path.join(TEST, "u10.npy")).astype(np.float32)
t_v = np.load(os.path.join(TEST, "v10.npy")).astype(np.float32)
t_p = np.load(os.path.join(TEST, "pblh.npy")).astype(np.float32)
t_t = np.load(os.path.join(TEST, "t2.npy")).astype(np.float32)

X_test_std = (np.stack([t_c, t_u, t_v, t_p, t_t], axis=-1) - stats[0]) / stats[1]
X_input = X_test_std.transpose(0, 4, 1, 2, 3).reshape(-1, 50, 140, 124)

with torch.no_grad(), torch.amp.autocast('cuda'):
    preds_delta = model(torch.from_numpy(X_input).to(device)).cpu().numpy()

# Reconstruction
last_known = X_test_std[:, 9, :, :, 0]
final_std = np.expand_dims(last_known, axis=1) + preds_delta
final_raw = (final_std * stats[1][0]) + stats[0][0]
final_raw = np.clip(final_raw, 0, None).transpose(0, 2, 3, 1)

np.save('/kaggle/working/preds.npy', final_raw.astype(np.float32))
print("Saved! Shape:", final_raw.shape)

Computing Z-Score Stats...
Starting Fast Training...
Epoch 1 | Loss: 0.06614
Epoch 2 | Loss: 0.05096
Epoch 3 | Loss: 0.04542
Epoch 4 | Loss: 0.04217
Epoch 5 | Loss: 0.03987
Epoch 6 | Loss: 0.03737
Epoch 7 | Loss: 0.03565
Epoch 8 | Loss: 0.03392
Epoch 9 | Loss: 0.03280
Epoch 10 | Loss: 0.03167
Epoch 11 | Loss: 0.03021
Epoch 12 | Loss: 0.02917
Epoch 13 | Loss: 0.02855
Epoch 14 | Loss: 0.02753
Epoch 15 | Loss: 0.02693
